# Chien luoc da khung thoi gian CK Yfinance
### Mot tin hieu mua xay ra khi H1 co MA9 < Close va D1: MA5 > MA21
### Mot tin hieu ban xay ra khi H1 co MA9 > Close va D1: MA5 < MA21

# Load data

In [ ]:
import sys
sys.path.append("../Common")
import CommonYFinance

In [3]:
import talib

# Import các thư viện cần thiết
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

def detectSignal(symbol, from_date, to_date, timeframe):

	import pandas as pd
	import plotly.graph_objects as go
	import redis
	import numpy as np
	from datetime import datetime

	# ##############################################Step 1: Lấy dữ liệu##############################################
	dataH1 = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, timeframe)
	dataD1 = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, "1d")
	
	# Mot tin hieu mua xay ra khi H1 co MA9 < Close va D1: MA5 > MA21
	# Mot tin hieu ban xay ra khi H1 co MA9 > Close va D1: MA5 < MA21
	dataH1['MA9'] = talib.SMA(dataH1['Close'], timeperiod=9)	

	# ML
	# Tạo một đặc trưng mới có thể là ngày (hoặc một đặc trưng khác) để sử dụng trong mô hình
	# Trong ví dụ này, giả sử bạn muốn sử dụng giá mở (Open) làm đặc trưng
	X = dataH1[['Open']] # Features O, H, L, C, V tron tu 1 bien den 5 bien
	# Sử dụng cột 'Close' làm target => Gia dong cua ngay hom qua, quyet gia dong cua ngay hom
	y = dataH1['Close']
	# Khởi tạo và huấn luyện mô hình cây quyết định
	# Mo hinh Decision Tree Regressor
	# Chia dữ liệu thành tập huấn luyện và tập kiểm thử
	model = DecisionTreeRegressor()
	model.fit(X, y)
	predictions = model.predict(X)

	dataH1['Predicted_Close'] = predictions
	
	dataH1['Buy_Signal'] = (dataH1['MA9'] < dataH1['Close']) & (dataH1['Predicted_Close'] > dataH1['Close'])
	dataH1['Sell_Signal'] = (dataH1['MA9'] > dataH1['Close']) & (dataH1['Predicted_Close'] < dataH1['Close'])

	dataD1['MA5'] = talib.SMA(dataD1['Close'], timeperiod=5)
	dataD1['MA21'] = talib.SMA(dataD1['Close'], timeperiod=21)

	dataD1['Buy_Signal'] = (dataD1['MA5'] > dataD1['MA21']) 
	dataD1['Sell_Signal'] = (dataD1['MA5'] < dataD1['MA21'])

	print(dataH1.iloc[-2])
	print(dataD1.iloc[-2])
	
	last_record = dataH1.iloc[-2]
	last_record['Buy_Signal'] = dataH1['Buy_Signal'].iloc[-2] & dataD1['Buy_Signal'].iloc[-2]
	last_record['Sell_Signal'] = dataH1['Sell_Signal'].iloc[-2] & dataD1['Sell_Signal'].iloc[-2]
		
	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line     Buy_Signal      Sell_Signal
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=0)
	# Tạo hash key
	hash_key = 'OG_YFINANCE_MULTIPLE_TIMEFRAME_H1_D1'
	   
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		print(last_record)   
	else:
		print(last_record)
		print('Không có tín hiệu!')

	# ##############################################Step 4: Vẽ biểu đồ##############################################  
	
	# import plotly.graph_objects as go

	# # Đảm bảo rằng data['Datetime'] là dạng datetime nếu chưa phải
	# # data['Datetime'] = pd.to_datetime(data['Datetime'])

	# # Tạo biểu đồ Candlestick cho dữ liệu giá
	# fig = go.Figure(data=[go.Candlestick(x=data['Datetime'],  # Sử dụng cột Datetime thay vì index
	#                                     open=data['Open'],
	#                                     high=data['High'],
	#                                     low=data['Low'],
	#                                     close=data['Close'],
	#                                     name='Candlestick')])

	# # Thêm dòng dự đoán giá đóng cửa từ mô hình hồi quy
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

	# # Thêm điểm mua
	# fig.add_trace(go.Scatter(x=data[data['Buy_Signal']]['Datetime'],
	#                         y=data[data['Buy_Signal']]['Close'],
	#                         mode='markers',
	#                         marker=dict(color='Green', size=10, symbol='triangle-up'),
	#                         name='Buy Signal'))

	# # Thêm điểm bán
	# fig.add_trace(go.Scatter(x=data[data['Sell_Signal']]['Datetime'],
	#                         y=data[data['Sell_Signal']]['Close'],
	#                         mode='markers',
	#                         marker=dict(color='Red', size=10, symbol='triangle-down'),
	#                         name='Sell Signal'))

	# # Thêm Momentum và RSI vào trục Y phụ
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['Momentum'], name='Momentum', yaxis='y2'))
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['RSI'], name='RSI', yaxis='y3'))

	# # Tùy chỉnh layout để thêm trục Y phụ và cập nhật tiêu đề trục X
	# fig.update_layout(
	#     title=f'Trading Signals for {symbol} based on Linear Regression, Momentum, and RSI',
	#     xaxis_title='Date',
	#     yaxis_title='Price',
	#     xaxis_rangeslider_visible=False,
	#     yaxis=dict(
	#         title='Price',
	#         titlefont=dict(color='#1f77b4'),
	#     ),
	#     yaxis2=dict(
	#         title='Momentum',
	#         titlefont=dict(color='#ff7f0e'),
	#         anchor='free',
	#         overlaying='y',
	#         side='left',
	#         position=0.15
	#     ),
	#     yaxis3=dict(
	#         title='RSI',
	#         titlefont=dict(color='#d62728'),
	#         anchor='x',
	#         overlaying='y',
	#         side='right'
	#     ),
	# )

	# fig.show()


In [4]:
from datetime import datetime, timedelta
import schedule
import time
import MetaTrader5 as mt5

def schedule_detectSignal():

	symbol = 'ACB.VN'
	from_date = (datetime.now() - timedelta(days=40)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
	to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	detectSignal(symbol, from_date, to_date, "1h")

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0,60,1)) # Tu 0 den 59
while True:
	current_time = datetime.now()
	current_minute = current_time.minute
	current_second = current_time.second
	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes: # Nếu có, gọi hàm detectSignal
		if current_second == 3: # Chỉ chạy hàm khi giây là 0
			# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
			last_run = getattr(schedule_detectSignal, 'last_run', None) # Lấy phút cuối cùng mà hàm đã chạy
			# Nếu chưa chạy trong phút hiện tại, gọi hàm và lưu lại phút hiện tại
			if last_run is None or last_run != current_minute:
				schedule_detectSignal()
				# Lưu lại phút cuối cùng mà hàm đã chạy
				setattr(schedule_detectSignal, 'last_run', current_minute)

	# Nghỉ 1 giây trước khi kiểm tra lại
	time.sleep(1)

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['ACB.VN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['ACB.VN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by DecisionTreeRegressor.